In [122]:
import sys
sys.path.append('..')

from utilities_amigocloud import AmigocloudFunctions
from sqlalchemy import create_engine, text
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from shapely import wkb

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [123]:
# conexion a base de datos
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# obtener datos de todo catastro
def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

# eliminar todas las propiedades cuyo unidad_01 este en la lista recibida
def eliminar_props(lista_cod):
    cod_string = ", ".join(map(str, lista_cod))
    query = f"""DELETE FROM dataset_390174 WHERE unidad_01 in ({cod_string})"""
    select = amigocloud.ejecutar_query_sql(ID_PROYECTO, query, 'post')
    return select

# funcion auxiliar para convertir poligono en multipoligono
def convertir_a_multipolygon(geometry):
    if isinstance(geometry, Polygon):
        return MultiPolygon([geometry])
    return geometry

# convertir poligono en formato wkb
def convertir_a_wkb(polygon):
    wkb_data = wkb.dumps(polygon, hex=True)
    return wkb_data

# cargar lotes de geodataframe a amigocloud de forma masiva
def cargar_props_a_amigocloud_bulk(gdf):
    # 1. Preparación de datos y reproyección
    gdf_gral = gdf.to_crs(epsg=4326).copy()
    # Asegurar que usamos la columna de geometría correcta
    # Si tu columna se llama 'geom', la procesamos así:
    gdf_gral['geometry_processed'] = gdf_gral['geom'].apply(convertir_a_multipolygon)
    # --- FUNCIÓN AUXILIAR DE LIMPIEZA ---
    def sql_v(val, tipo='num'):
        """
        Convierte valores de Python/Pandas a sintaxis SQL válida.
        tipo: 'num', 'str', 'date'
        """
        if pd.isna(val) or val is None or str(val).lower() == 'nan':
            return "NULL"
        if tipo == 'str':
            # Quitar espacios extras y escapar comillas simples
            texto = str(val).strip().replace("'", "''")
            return f"'{texto}'"
        if tipo == 'date':
            return f"'{val}'"
            
        # Para números (int, float)
        return str(val)
    # 2. Construir la lista de valores para el SQL
    valores_rows = []
    for index, row in gdf_gral.iterrows():
        # Generar el WKB Hexadecimal
        wkb_hex = convertir_a_wkb(row['geometry_processed'])
        # Formateamos la fila usando la función sql_v
        # El orden debe ser idéntico al INSERT del paso 4
        fila_sql = f"""(
            ST_SetSRID(ST_GeomFromWKB('\\x{wkb_hex}'), 4326),
            {sql_v(row.get('idd'))},
            {sql_v(row.get('unidad_01'))},
            {sql_v(row.get('unidad_02'), 'str')},
            {sql_v(row.get('unidad_03'))},
            {sql_v(row.get('unidad_04'), 'str')},
            {sql_v(row.get('unidad_05'), 'str')},
            {sql_v(row.get('variedad'), 'str')},
            {sql_v(row.get('soca'))},
            {sql_v(row.get('textura'), 'str')},
            {sql_v(row.get('cultivo'), 'str')},
            {sql_v(row.get('financia'), 'str')},
            {sql_v(row.get('fs'), 'date')},
            {sql_v(row.get('area'))},
            {sql_v(row.get('fc'), 'date')},
            {sql_v(row.get('zafra'))},
            {sql_v(row.get('lote_admin'), 'str')}
        )"""
        valores_rows.append(fila_sql)

    # 3. Unir todas las filas
    todos_los_valores = ",\n".join(valores_rows)
    # 4. Crear el Query Final
    insert_sql = f"""
        INSERT INTO dataset_390174
        (wkb_geometry, idd, unidad_01, unidad_02, unidad_03, unidad_04, unidad_05, variedad, soca, textura, cultivo, financia, fs, area, fc, zafra, lote_admin)
        VALUES 
        {todos_los_valores};
    """
    # 5. Ejecutar
    try:
        print(f"Enviando {len(gdf_gral)} registros a AmigoCloud (Dataset 390174)...")
        resultado = amigocloud.ejecutar_query_sql(ID_PROYECTO, insert_sql, 'post') 
        if resultado:
            print("✅ Carga masiva exitosa.")
            # Opcional: marcar como planificado localmente
            # for l_id in gdf_gral['idd']: marcar_lote_como_planificado(l_id)
        else:
            print("❌ El servidor no confirmó la carga.")
            
    except Exception as e:
        print(f"❌ Error en el Insert Masivo: {e}")
    return None

In [124]:
ID_PROYECTO = 34054
amigocloud = AmigocloudFunctions(token=API_AMIGOCLOUD_TOKEN_ADM)
amigocloud

In [125]:
# get todos los datos de catastro
gdf_catastro = get_catastro()

In [126]:
# lista de codigos de propiedad a eliminar
lista_props = [
1520
]

In [127]:
# eliminar propiedades
eliminar_props(lista_props)

{'query': 'DELETE FROM dataset_390174 WHERE unidad_01 in (1520)',
 'count': 44,
 'amigo_ids': ['64a4e24da81849ecabd512f7ac4abc5b',
  '86dff08c89f1486cb1eb43865f17f0a0',
  '94cd62511e4245a0aa3daae120ee255e',
  '757c4e90da184b6ca0343b4ea1b718e5',
  '76aee0ec414547bda3e7601107a4fbe8',
  'ec0db72ab96046afa5b97716cc428210',
  '48001502d98b4905a0322f6b0ec77c5e',
  'f54fec17ed3e4659b6070df90fa9729a',
  '8c896a7d9a694f82b4bde023977e17f0',
  '6bce957d785f4fff927bdaeb6e805c51',
  'de4055586e2d4a4a9ed026a397d86bd8',
  '3e7ce96f65334f3c9333a0b73d54a8e5',
  '183fdebe43d2442ba3c8d89063f3da6e',
  '4dec532033db47409457e161f0a2028d',
  'ba38eeb24d3445d39e4557e6c0f8170c',
  'ac7ec7150e004d2ca7ad798e6646aa70',
  'cad92845fa6949c28520eabb8be69494',
  '91f9c0983dff4d66af802249ec6fac0d',
  'fc737768509a4e44ac1c9578e28ce3a7',
  'e2eb6a2e45e4488097ecba6d8dff89a2',
  '10d44cdf64d44f83bc6ff88eb27317b7',
  '845ba3fff3284a96bd01177627b22a38',
  'f652637455c94595956b3021fa0eae1f',
  'af9add38345043f88ce5febede22aa

In [128]:
# leccionar propiedades segun lista de codigos
props_select_segun_lista = gdf_catastro[gdf_catastro['unidad_01'].isin(lista_props)]

In [129]:
# cargar gdf a amigocloud de forma masiva
cargar_props_a_amigocloud_bulk(props_select_segun_lista)

Enviando 44 registros a AmigoCloud (Dataset 390174)...
✅ Carga masiva exitosa.
